In [1]:
# %%
import os
import re
import shutil
import neo
import logging
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import lightgbm as lgb
from scipy.signal import savgol_filter, find_peaks,medfilt
from scipy.fft import fft, fftfreq
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from scipy import sparse,stats
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
from numba import jit
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn import datasets
from mlxtend.classifier import StackingClassifier
from collections import defaultdict
from scipy.sparse.linalg import spsolve
import seaborn as sns
import pywt
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve,classification_report,confusion_matrix
from sklearn.model_selection import StratifiedKFold,cross_val_score,GroupKFold,StratifiedKFold,LeaveOneGroupOut
from scipy.stats import mode,zscore
from collections import Counter
pd.set_option('display.max_colwidth', 500)
pd.set_option('display.max_columns', None)

from scipy.signal import welch
from sklearn.decomposition import PCA
import nolds 
import antropy as ant
from python_speech_features import mfcc
from antropy import spectral_entropy

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.utils.data
from torch.utils.data import DataLoader, Dataset
import numpy as np
from scipy.signal import decimate
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from antropy import spectral_entropy
from torch.nn.parallel import DataParallel
tmp = "0,2,3,4,5,6,7"
os.environ["CUDA_VISIBLE_DEVICES"] = tmp




# %%
def abf_numpy(addr):
    reader = neo.io.AxonIO(filename=addr)
    block = reader.read_block()
    segment = block.segments[0]
    signal = segment.analogsignals[0]
    currency_data = signal.magnitude
    if currency_data.shape[-1] >1 :
        currency_data = currency_data[:,0]
    currency_data = currency_data.flatten()
    return currency_data
def wavelet_baseline(signal, wavelet='db4', level=3):
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    coeffs1 = coeffs.copy()
    coeffs1[1:] = [np.zeros_like(c) for c in coeffs1[1:]]
    baseline = pywt.waverec(coeffs1, wavelet)
    return baseline[:len(signal)],coeffs

def signal_normalize_whole(current_smooth):
    current_normalized = current_smooth - np.mean(current_smooth)  # 去除直流偏移
    current_normalized = (current_normalized - np.min(current_normalized)) / (np.max(current_normalized) - np.min(current_normalized))
    return current_normalized
def signal_normalize_half(current_smooth):
    current_normalized = (current_smooth - np.min(current_smooth)) / (np.max(current_smooth) - np.min(current_smooth))
    return current_normalized
def cut_slide(currency_data,position,target_length):
    ###########################target_length = 200001
    target_length = (target_length // 3) * 3
    currency_data1 = currency_data[((currency_data.shape[0]//10)*position):((currency_data.shape[0]//10)*position)+target_length]
    currency_data1 = currency_data1[:, np.newaxis]
    currency_data1 = currency_data1.flatten()
    return currency_data1
def robust_normalize(x):
    """鲁棒归一化：映射到[0,1]范围"""
    med = np.median(x)
    iqr = stats.iqr(x)
    # 防止除零
    if iqr < 1e-8:
        return np.zeros_like(x)
    # 缩放到IQR的4倍范围（覆盖约95%正态数据）
    normalized = (x - med) / (4 * iqr) + 0.5
    # 裁剪到[0,1]
    #return np.clip(normalized, 0, 1)
    #不裁剪了
    return normalized
def katz_fractal_dimension(signal):
    dists = np.abs(np.diff(signal))
    ll = np.sum(dists)
    avg_dist = np.mean(dists)
    d_max = np.max(np.abs(signal - np.roll(signal, 1)))
    kfd = np.log10(ll) / (np.log10(d_max) + np.log10(avg_dist))
    return kfd

# %%
class CWTProcessor:
    def __init__(self, scales, sampling_rate=1e6):
        """优化后的CWT处理器，支持多GPU并行"""
        self.scales = scales
        self.sampling_rate = sampling_rate
        
    def _morlet_kernel(self, scale, device):
        """生成指定尺度的Morlet小波核"""
        dt = 1.0 / self.sampling_rate
        time = torch.arange(-5*scale, 5*scale + dt, dt, device=device)
        wavelet = torch.exp(1j*2*3.141592653589793*time/scale) * torch.exp(-time**2/(2*scale**2))
        return wavelet / torch.norm(wavelet)

    def process_chunk(self, chunk, device):
        """处理单个数据块"""
        torch.cuda.set_device(device)
        chunk_tensor = torch.tensor(chunk, dtype=torch.float32, device=device)
        n_scales = len(self.scales)
        energies = []
        chunk_real = chunk_tensor.real if torch.is_complex(chunk_tensor) else chunk_tensor
        chunk_fft = torch.fft.rfft(chunk_real)

        for scale in self.scales:
            kernel = self._morlet_kernel(scale, device)
            kernel_real = kernel.real
            kernel_fft = torch.fft.rfft(kernel_real, n=len(chunk_real))
            conv = torch.fft.irfft(chunk_fft * kernel_fft, n=len(chunk_real))
            energy = conv[-len(chunk_real):].abs().square()
            energies.append(energy.cpu().numpy())  # 立即转移到CPU
            del kernel, kernel_real, kernel_fft, conv, energy
            torch.cuda.empty_cache()
        del chunk_fft, chunk_real, chunk_tensor
        torch.cuda.empty_cache()
        return np.stack(energies)


class ChunkDataset(Dataset):
    """优化数据加载"""
    def __init__(self, chunks):
        self.chunks = [np.array(chunk, dtype=np.float32) for chunk in chunks]
        
    def __len__(self):
        return len(self.chunks)
    
    def __getitem__(self, idx):
        return self.chunks[idx]

def distributed_cwt(signal, fs=5e5, num_gpus=5):                     
    # === 1. 信号预处理 ===
    downsample_factor = 40  # 进一步降低降采样率以减少数据量
    signal_down = decimate(signal, downsample_factor, ftype='fir')
    
    # === 2. 数据分块 ===
    chunk_size = 100_000  # 进一步减小分块大小
    chunks = [signal_down[i:i+chunk_size] 
             for i in range(0, len(signal_down), chunk_size)]
    
    # === 3. 初始化多GPU处理 ===
    devices = [f"cuda:{i}" for i in range(num_gpus)]
    
    # === 4. 创建处理器 ===
    scales = np.logspace(0, 2.5, 20, base=10)  # 进一步减少尺度数量
    processor = CWTProcessor(scales, sampling_rate=fs/downsample_factor)
    
    # === 5. 并行处理 ===
    features = []
    with ThreadPoolExecutor(max_workers=num_gpus) as executor:
        futures = []
        for i, chunk in enumerate(chunks):
            device = devices[i % num_gpus]
            futures.append(executor.submit(
                processor.process_chunk, chunk, device))
            
        for future in futures:
            energy = future.result()
            # 提取特征
            chunk_features = {
                'max_energy': energy.max(),
                'mean_energy': energy.mean(),
                'entropy': spectral_entropy(energy.mean(axis=0), sf=fs)
            }
            features.append(chunk_features)
    
    return features

# %%
def process_abf_file(file_path,isminus=False,wavelet='db4', level=3, sg_window=31, sg_order=5):
    try:
        # 1. 数据加载与预处理
        raw_data = abf_numpy(file_path)
        signal = raw_data
        if isminus:
            signal = signal*(-1)

        baseline, coeffs = wavelet_baseline(signal, 'db4', 3)
        corrected = signal - baseline
        corrected_sg = savgol_filter(corrected, 31, 5)
        processed = np.where(corrected < 0, 0, corrected)
        processed = robust_normalize(processed)
        corrected_sg = signal_normalize_whole(corrected_sg)
        


        # 预处理信号
        features = {}
        processed = np.where(signal - baseline < 0, 0, signal - baseline)
        processed = robust_normalize(processed)

        # === 时域特征 ===
        peaks, _ = find_peaks(processed, height=0.5)

        # 原有特征
        features.update({
            'peak_count': len(peaks),
            'peak_mean': np.mean(processed[peaks]) if len(peaks) > 0 else 0,
            'peak_std': np.std(processed[peaks]) if len(peaks) > 0 else 0,
            'signal_std': np.std(processed),
            'signal_skew': stats.skew(processed),
            'signal_kurtosis': stats.kurtosis(processed),
            'signal_median': np.median(corrected_sg),
            'rms': np.sqrt(np.mean(np.square(corrected_sg))),
        })

        # 新增时域特征
        hjorth_mobility = np.sqrt(np.var(np.gradient(corrected_sg)) / np.var(corrected_sg))
        features.update({
            # Hjorth参数
            'hjorth_activity': np.var(corrected_sg),
            'hjorth_mobility': hjorth_mobility,
            'hjorth_complexity': (np.var(np.gradient(np.gradient(corrected_sg)) / np.var(np.gradient(corrected_sg))) / hjorth_mobility)
        })


        features.update({    
            # 分形维度
            'kfd': katz_fractal_dimension(corrected_sg)
        })

        # 频域特征
        N = len(corrected_sg)
        T = 2e-6
        yf = fft(corrected_sg)
        xf = fftfreq(N, T)[:N//2]
        magnitudes = np.abs(yf[:N//2]) * 2 / N
        sorted_indices = np.argsort(magnitudes)[::-1]  # 降序排列
        top_freq_indices = sorted_indices[:3]
        features.update({
            'dominant_freq1': xf[top_freq_indices[0]],
            'dominant_freq2': xf[top_freq_indices[1]] if len(top_freq_indices)>1 else 0,
            'dominant_freq3': xf[top_freq_indices[2]] if len(top_freq_indices)>2 else 0,
            'spectral_entropy': -np.sum(magnitudes * np.log(magnitudes + 1e-10)),
            'total_power': np.sum(magnitudes**2),
            'max_power': np.max(magnitudes**2)
        })

        freqs, psd = welch(corrected_sg, fs=fs, nperseg=1024)
        sorted_indices = np.argsort(psd)[::-1]
        top_freq_indices = sorted_indices[:3]

        features.update({
            'dominant_freq1_Wel': freqs[top_freq_indices[0]],
            'dominant_freq2_Wel': freqs[top_freq_indices[1]] if len(top_freq_indices)>1 else 0,
            'dominant_freq3_Wel': freqs[top_freq_indices[2]] if len(top_freq_indices)>2 else 0,
            'spectral_entropy_Wel': ant.spectral_entropy(corrected_sg, sf=fs, method='welch'),
            'total_power_Wel': np.sum(psd),
            'max_power_Wel': np.max(psd),  # 建议改为max_psd更准确
            # 功率谱质心
            'spectral_centroid': np.sum(freqs * psd) / np.sum(psd)
        })

        # 新增频域特征
        #Mfcc = mfcc(signal=corrected_sg, samplerate=fs, numcep=3,nfft=12500)

        '''features.update({
            # MFCC (取前5个系数)
            'mfcc1': np.mean(Mfcc[:,0]),
            'mfcc2': np.mean(Mfcc[:,1]),
            'mfcc3': np.mean(Mfcc[:,2]),

        })'''



        # === 小波特征 ===
        features.update({
            'wavelet_energy': np.sum(np.square(coeffs[-1])),
            'wavelet_subband1': np.sum(np.square(coeffs[0])),
            'wavelet_subband2': np.sum(np.square(coeffs[1])),
            'wavelet_subband3': np.sum(np.square(coeffs[2])),
        })

        '''features_CWT = distributed_cwt(corrected_sg)
        max_energies = [f['max_energy'] for f in features_CWT]
        mean_energies = [f['mean_energy'] for f in features_CWT]
        entropies = [f['entropy'] for f in features_CWT]
        # 计算全局统计量
        global_features_CWT = {
            'CWT_max_energy_max': np.max(max_energies),
            'CWT_max_energy_median': np.median(max_energies),
            'CWT_mean_energy_max': np.max(mean_energies),
            'CWT_mean_energy_median': np.median(mean_energies),
            'CWT_entropy_max': np.max(entropies),
            'CWT_entropy_median': np.median(entropies)
        }
        features.update(global_features_CWT)'''
        
        return features
    
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return None
def batch_process_abf_files(file_list,isminus=False):
    results = []
    for file_path in tqdm(file_list, desc="Processing files"):
        features = process_abf_file(file_path,isminus=False)
        if features is not None:
            features['source'] = file_path
            results.append(features)
    return pd.DataFrame(results)


In [2]:
data_index = []
for root,dirs,files in os.walk('/data4/baihexiang/fangshui/'):
    for file in files:
        data_index.append(os.path.join(root,file))
data_index_plus = []
data_index_minus = []

In [3]:
data_index

['/data4/baihexiang/fangshui/131-zhengchangren/fangshui.xlsx',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/20260131_na5_0000.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0013.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0012.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0011.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0010.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0009.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0008.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/-300/15_0007.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/+300/15_0006.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/+300/15_0005.abf',
 '/data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15/+300/15_0004.abf',
 '/data4/baihexiang/fangshui/131-

In [4]:
data_index_plus = []
data_index_minus = []

for item in data_index:
    folder = item.split('/')[-2]   # 取倒数第二级目录名
    if item.endswith('.abf'):
    # 先判断“正类”
        if ('+' in folder) or ('正' in folder):
            data_index_plus.append(item)
        elif ('-' in folder) or ('负' in folder):
            data_index_minus.append(item)

In [7]:
# 后面逻辑不变
fs=5e5
df_result = batch_process_abf_files(data_index_plus)
df_result_minus = batch_process_abf_files(data_index_minus, isminus=True)

df_result.to_csv('/data4/baihexiang/cache/df_result_fangshui.csv', index=None)
df_result_minus.to_csv('/data4/baihexiang/cache/df_result_fangshui_minus.csv', index=None)

Processing files: 100%|██████████| 233/233 [53:25<00:00, 13.76s/it]


In [11]:
df_result['source'].apply(lambda x:('/').join(x.split('/')[:-2]))

0      /data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15
1      /data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15
2      /data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15
3      /data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15
4      /data4/baihexiang/fangshui/131-zhengchangren/20260131_na5/15
                                   ...                             
214        /data4/baihexiang/fangshui/128/famgshui/20260128_na1/172
215        /data4/baihexiang/fangshui/128/famgshui/20260128_na1/172
216        /data4/baihexiang/fangshui/128/famgshui/20260128_na1/172
217        /data4/baihexiang/fangshui/128/famgshui/20260128_na1/172
218        /data4/baihexiang/fangshui/128/famgshui/20260128_na1/172
Name: source, Length: 219, dtype: object

In [13]:
def merge_df(df_result,df_result_minus):
    df_result['样本'] = df_result['source'].apply(lambda x:('/').join(x.split('/')[:-2]))
    df_result.dropna(inplace=True)
    df_result.reset_index(drop=True,inplace=True)

    df_result_minus['样本'] = df_result_minus['source'].apply(lambda x:('/').join(x.split('/')[:-2]))
    df_result_minus.dropna(inplace=True)
    df_result_minus = df_result_minus.add_suffix('_minus')
    df_result_minus.reset_index(drop=True,inplace=True)

    df_merge = pd.DataFrame()
    for item in set(df_result['样本']):
        df_slide1 = df_result[df_result['样本']==item].reset_index()
        df_slide2 = df_result_minus[df_result_minus['样本_minus']==item].reset_index()
        df_slide3 = pd.concat([df_slide1,df_slide2],axis=1)
        df_merge = pd.concat([df_merge,df_slide3],axis=0)
    df_merge.drop(columns=['index'],inplace=True)
    df_merge.dropna(inplace=True)
    df_merge.reset_index(drop=True,inplace=True)
    return df_merge

In [14]:
df_merge = merge_df(df_result,df_result_minus)

In [15]:
df_merge

,peak_count,peak_mean,peak_std,signal_std,signal_skew,signal_kurtosis,signal_median,rms,hjorth_activity,hjorth_mobility,hjorth_complexity,kfd,dominant_freq1,dominant_freq2,dominant_freq3,spectral_entropy,total_power,max_power,dominant_freq1_Wel,dominant_freq2_Wel,dominant_freq3_Wel,spectral_entropy_Wel,total_power_Wel,max_power_Wel,spectral_centroid,wavelet_energy,wavelet_subband1,wavelet_subband2,wavelet_subband3,source,样本,peak_count_minus,peak_mean_minus,peak_std_minus,signal_std_minus,signal_skew_minus,signal_kurtosis_minus,signal_median_minus,rms_minus,hjorth_activity_minus,hjorth_mobility_minus,hjorth_complexity_minus,kfd_minus,dominant_freq1_minus,dominant_freq2_minus,dominant_freq3_minus,spectral_entropy_minus,total_power_minus,max_power_minus,dominant_freq1_Wel_minus,dominant_freq2_Wel_minus,dominant_freq3_Wel_minus,spectral_entropy_Wel_minus,total_power_Wel_minus,max_power_Wel_minus,spectral_centroid_minus,wavelet_energy_minus,wavelet_subband1_minus,wavelet_subband2_minus,wavelet_subband3_minus,source_minus,样本_minus
0,4846983.0,0.977156,0.262887,0.232264,1.672777,2.329751,0.519076,0.526142,0.007406,0.448884,559.725769,-3.496377,0.0,22500.000000,27343.750000,2363.394100,1.092488,1.077675,28320.3125,27832.03125,28808.59375,4.540385,0.000015,4.389091e-07,37224.518789,35009820.0,4.883081e+16,41935904.0,33955544.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/+300/103_0005.abf,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103,4868544.0,0.979737,0.264374,0.233164,1.679110,2.357610,0.619527,0.622803,0.004124,0.448720,1005.514526,-3.287214,0.0,22500.000000,27343.750000,1813.988050,1.543286,1.535039,28320.3125,28808.59375,27832.03125,4.537477,0.000008,2.439096e-07,37235.509017,35566972.0,5.575028e+16,42106240.0,33994216.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/-300/103_0016.abf,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103
1,4848445.0,0.978230,0.263151,0.232582,1.672501,2.327790,0.502202,0.510193,0.008120,0.449309,509.144592,-3.677339,0.0,27343.750000,31250.000000,2469.670888,1.024947,1.008708,28320.3125,27832.03125,29296.87500,4.539793,0.000017,4.810515e-07,37276.664481,35070868.0,4.813619e+16,41653824.0,33819508.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/+300/103_0004.abf,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103,4883987.0,0.979504,0.264162,0.233036,1.678136,2.351949,0.566579,0.569029,0.002811,0.448538,1472.909058,-3.073696,0.0,28468.183333,27343.750000,1524.340135,1.289554,1.283932,28320.3125,27832.03125,27343.75000,4.537101,0.000006,1.661065e-07,37226.565695,35607436.0,5.524670e+16,42169756.0,33761896.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/-300/103_0015.abf,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103
2,4886623.0,0.977279,0.263796,0.232839,1.681052,2.393236,0.470813,0.476464,0.005390,0.448681,767.749146,-3.393906,0.0,27343.750000,28468.183333,2048.306249,0.897293,0.886513,28320.3125,27832.03125,27343.75000,4.533337,0.000011,3.217075e-07,37220.717421,35576040.0,4.745990e+16,42410548.0,34152644.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/+300/103_0003.abf,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103,4876490.0,0.980074,0.264257,0.233230,1.677621,2.348651,0.536622,0.543560,0.007539,0.449032,549.469604,-2.995918,0.0,22500.000000,27343.750000,2384.693063,1.166749,1.151670,28320.3125,27832.03125,27343.75000,4.540890,0.000015,4.465876e-07,37273.953402,35735984.0,5.504734e+16,42030600.0,33945068.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/-300/103_0014.abf,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103
3,4877080.0,0.976552,0.261799,0.231936,1.668999,2.312250,0.562174,0.568066,0.006698,0.449261,617.381042,-3.481606,0.0,27343.750000,31250.000000,2263.352442,1.277398,1.264002,28320.3125,27832.03125,28808.59375,4.538510,0.000014,3.976916e-07,37282.453983,35222288.0,4.679687e+16,41650420.0,33829012.0,/data4/baihexiang/fangshui/128/famgshui/20260128_na7/103/+300/103_0002.abf,/da

In [18]:
df_merge_all = df_merge.copy()
df_merge_all['label'] = df_merge_all['source'].str.contains('zhengchangren').astype(int)

In [19]:
df_merge3 = df_merge_all.copy()
df_merge3['bag'] = df_merge3['样本']
y = df_merge3['label']
X = df_merge3.drop(columns=['label','bag','source','source_minus','样本','样本_minus'])

In [20]:
X.columns

Index(['peak_count', 'peak_mean', 'peak_std', 'signal_std', 'signal_skew',
       'signal_kurtosis', 'signal_median', 'rms', 'hjorth_activity',
       'hjorth_mobility', 'hjorth_complexity', 'kfd', 'dominant_freq1',
       'dominant_freq2', 'dominant_freq3', 'spectral_entropy', 'total_power',
       'max_power', 'dominant_freq1_Wel', 'dominant_freq2_Wel',
       'dominant_freq3_Wel', 'spectral_entropy_Wel', 'total_power_Wel',
       'max_power_Wel', 'spectral_centroid', 'wavelet_energy',
       'wavelet_subband1', 'wavelet_subband2', 'wavelet_subband3',
       'peak_count_minus', 'peak_mean_minus', 'peak_std_minus',
       'signal_std_minus', 'signal_skew_minus', 'signal_kurtosis_minus',
       'signal_median_minus', 'rms_minus', 'hjorth_activity_minus',
       'hjorth_mobility_minus', 'hjorth_complexity_minus', 'kfd_minus',
       'dominant_freq1_minus', 'dominant_freq2_minus', 'dominant_freq3_minus',
       'spectral_entropy_minus', 'total_power_minus', 'max_power_minus',
       'dom

In [21]:
from collections import defaultdict
import numpy as np
from scipy.stats import mode
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold  # 修改为GroupKFold
from tqdm import tqdm

# 初始化
bag_predictions = defaultdict(list)
bag_probabilities = defaultdict(list)
bag_true_labels = {}
bag_info = []

true_labels = []
predictions = []

# 新增：样本级软投票预测
sample_soft_preds = []

# 训练器和分组器
superdog = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=1,
    random_seed=2018,
    loss_function='Logloss',
    thread_count=100,
    verbose=0
)
# 修改为10折分组交叉验证
gkf = GroupKFold(n_splits=10)  # 使用10折
groups = df_merge3['bag'].astype('category').cat.codes.values

# Cross-Validation
with tqdm(total=gkf.get_n_splits(X, y, groups), desc="Cross-Validating") as pbar:
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):  # 修改为gkf.split
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        test_bags = df_merge3.iloc[test_idx]['bag'].values
        
        # 样本权重（可选）
        bag_sizes = df_merge3.groupby('bag').size()
        sample_weights = 1 / bag_sizes[df_merge3.iloc[train_idx]['bag']].values
        
        superdog.fit(X_train, y_train)
        preds = superdog.predict(X_test).flatten()
        probas = superdog.predict_proba(X_test)
        
        # 样本级硬投票记录
        true_labels.extend(y_test)
        predictions.extend(preds)
        
        # 收集 Bag 级数据
        for bag, true, pred, prob in zip(test_bags, y_test, preds, probas):
            bag_predictions[bag].append(pred)
            bag_probabilities[bag].append(prob)
            bag_true_labels[bag] = true
            bag_info.append({
                'bag': bag,
                'sample_true': true,
                'sample_pred_hard': pred,
                'sample_proba': prob
            })
        pbar.update(1)

# Bag 级聚合与评估
final_true = []
final_pred_hard = []
final_pred_soft = []
final_probas = []
bag_names = []

for bag in bag_predictions:
    # 真实标签一致性检查
    unique_true = np.unique([info['sample_true'] for info in bag_info if info['bag']==bag])
    assert len(unique_true)==1, f"Bag {bag} 真实标签冲突: {unique_true}"
    
    # 硬投票
    pred_mode, _ = mode(bag_predictions[bag])
    pred_hard = pred_mode.item() if hasattr(pred_mode, 'item') else pred_mode[0]
    
    # 软投票：平均概率后取 argmax
    prob_mean = np.mean(bag_probabilities[bag], axis=0)
    pred_soft = int(np.argmax(prob_mean))
    
    # 收集
    final_true.append(unique_true[0])
    final_pred_hard.append(pred_hard)
    final_pred_soft.append(pred_soft)
    final_probas.append(prob_mean)
    bag_names.append(bag)

# Bag 级硬投票评估
print("\nBag 级硬投票评估:")
print(classification_report(final_true, final_pred_hard))
print(f"Accuracy: {accuracy_score(final_true, final_pred_hard):.4f}")
print("Confusion Matrix:\n", confusion_matrix(final_true, final_pred_hard))

# Bag 级软投票评估
print("\nBag 级软投票评估:")
print(classification_report(final_true, final_pred_soft))
print(f"Accuracy: {accuracy_score(final_true, final_pred_soft):.4f}")

# AUC（多分类 OvR 或 二分类）
final_probas = np.array(final_probas)
if final_probas.shape[1] > 2:
    auc = roc_auc_score(final_true, final_probas, multi_class='ovr')
else:
    auc = roc_auc_score(final_true, final_probas[:, 1])
print(f"Bag 级软投票 AUC: {auc:.4f}")

# 样本级评估对比
print("\n样本级硬投票评估:")
print(classification_report(true_labels, predictions))

# 样本级软投票：将每个样本的预测替换为其所属 bag 的 soft-vote 结果
# 构建 bag->soft_pred 映射
bag_soft_map = dict(zip(bag_names, final_pred_soft))
for info in bag_info:
    sample_soft_preds.append(bag_soft_map[info['bag']])

print("\n样本级软投票评估:")
print(classification_report(true_labels, sample_soft_preds))
print(f"Accuracy: {accuracy_score(true_labels, sample_soft_preds):.4f}")

Cross-Validating:   0%|          | 0/10 [00:00<?, ?it/s]TBB Warning: The number of workers is currently limited to 39. The request for 99 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.

Cross-Validating: 100%|██████████| 10/10 [01:29<00:00,  8.99s/it]


Bag 级硬投票评估:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        29
           1       1.00      0.92      0.96        13

    accuracy                           0.98        42
   macro avg       0.98      0.96      0.97        42
weighted avg       0.98      0.98      0.98        42

Accuracy: 0.9762
Confusion Matrix:
 [[29  0]
 [ 1 12]]

Bag 级软投票评估:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        29
           1       1.00      0.92      0.96        13

    accuracy                           0.98        42
   macro avg       0.98      0.96      0.97        42
weighted avg       0.98      0.98      0.98        42

Accuracy: 0.9762
Bag 级软投票 AUC: 0.9920

样本级硬投票评估:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       148
           1       0.85      0.88      0.86        65

    accuracy                           0.92       21

In [2]:
import pandas as pd
df_info_final_merge = pd.read_csv('/data3/baihexiang/code/nm_code/FInal_fight/临床文章/After1_21/data/df_info_final_merge_1_26.csv')

In [4]:
df_info_final_merge.columns

Index(['peak_count', 'peak_mean', 'peak_std', 'signal_std', 'signal_skew',
       'signal_kurtosis', 'signal_median', 'rms', 'hjorth_activity',
       'hjorth_mobility', 'hjorth_complexity', 'kfd', 'dominant_freq1',
       'dominant_freq2', 'dominant_freq3', 'spectral_entropy', 'total_power',
       'max_power', 'dominant_freq1_Wel', 'dominant_freq2_Wel',
       'dominant_freq3_Wel', 'spectral_entropy_Wel', 'total_power_Wel',
       'max_power_Wel', 'spectral_centroid', 'wavelet_energy',
       'wavelet_subband1', 'wavelet_subband2', 'wavelet_subband3', 'source',
       '样本', 'peak_count_minus', 'peak_mean_minus', 'peak_std_minus',
       'signal_std_minus', 'signal_skew_minus', 'signal_kurtosis_minus',
       'signal_median_minus', 'rms_minus', 'hjorth_activity_minus',
       'hjorth_mobility_minus', 'hjorth_complexity_minus', 'kfd_minus',
       'dominant_freq1_minus', 'dominant_freq2_minus', 'dominant_freq3_minus',
       'spectral_entropy_minus', 'total_power_minus', 'max_power_min

In [5]:
df_info_final_merge.drop(columns=['性别',
       '年龄', 'BMI', 'CEA', 'AFP', 'CA199'],inplace=True)

In [6]:
df_info_final_merge.to_csv('/data4/baihexiang/NanoScaner/df_feature_default.csv',index=None)